## Quadratic Equation workflow

In [ ]:
from langchain_community.chat_models import ChatOllama

model = ChatOllama(model="llama3.2:1b")

model.invoke("Hello, how are you?").content

In [21]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field
from typing import Literal


In [22]:
# State of quadratic equation solver
class QuadraticState(BaseModel):
    a: float = Field( description="Coefficient of x^2")
    b: float = Field( description="Coefficient of x")
    c: float = Field( description="Constant term")
    equation: str = Field( default=None, description="The quadratic equation in string format")
    discriminant: float = Field( default=None, description="Discriminant value")
    results: str = Field( default=None, description="Roots of the equation or nature of roots")

In [36]:
# functions

# display equation
def show_equation(state: QuadraticState) -> dict:
    state.equation = f"Equation will be {state.a}x^2 + {state.b}x + {state.c}"
    return {'equation': state.equation}

# compute discriminant
def compute_discriminant(state: QuadraticState) -> dict:
    state.discriminant = state.b**2 - 4*state.a*state.c
    return {'discriminant': state.discriminant}

# determine nature of roots
def real_root(state: QuadraticState) -> dict:
    root1 = (-state.b + state.discriminant**0.5)/(2*state.a)
    root2 = (-state.b - state.discriminant**0.5)/(2*state.a)
    state.results = f"The roots are real and different: {root1} and {root2}"
    return {'results': state.results}

# determine nature of roots
def repeated_root(state: QuadraticState) -> dict:
    root = -state.b/(2*state.a)
    state.results = f"The roots are real and same: {root}"
    return {'results': state.results}

# determine nature of roots
def complex_root(state: QuadraticState) -> dict:
    real_part = -state.b / (2 * state.a)
    imaginary_part = round((abs(state.discriminant)**0.5) / (2 * state.a), 3)
    state.results = f"The roots are complex: {real_part} + {imaginary_part}i and {real_part} - {imaginary_part}i"
    return {'results': state.results}

def check_condition(state:QuadraticState)-> Literal['real_root','repeated_root','complex_root']:
    if state.discriminant > 0:
        return 'real_root'
    elif state.discriminant == 0:
        return 'repeated_root'
    else:
        return 'complex_root'

In [41]:
graph = StateGraph(QuadraticState)

# add nodes
graph.add_node('show_equation',show_equation)
graph.add_node('compute_discriminant',compute_discriminant)
graph.add_node('real_root',real_root)
graph.add_node('repeated_root',repeated_root)
graph.add_node('complex_root',complex_root)


# add edges

graph.add_edge(START,'show_equation')
graph.add_edge('show_equation','compute_discriminant')
graph.add_conditional_edges('compute_discriminant',check_condition)

graph.add_edge('real_root',END)
graph.add_edge('repeated_root',END)
graph.add_edge('complex_root',END)

workflow = graph.compile()


In [42]:
initial_state = {
    "a": 4, 
    "b": 4,
    "c": 4
}

final_state = workflow.invoke(initial_state)

In [43]:
final_state


{'a': 4,
 'b': 4,
 'c': 4,
 'equation': 'Equation will be 4.0x^2 + 4.0x + 4.0',
 'discriminant': -48.0,
 'results': 'The roots are complex: -0.5 + 0.866i and -0.5 - 0.866i'}